# Hugging Face Pipelines and Fine-Tuning
Let's finish the task of **Text classification** using IMDb sentiment analysis.

The goal is not only to run code, but also to understand the workflow:

- choose a pretrained model,
- load and preprocess a dataset,
- fine-tune the model,
- save the model,
- reload it with `pipeline()`,
- test the final system,
- discuss results and limitations.

## 0. Setup

Run the following cell first. In Google Colab, use **Runtime → Change runtime type → GPU** when possible.

The full datasets can be large, so this notebook uses **small subsets** for teaching purposes. Students may increase the subset size if they have enough compute.

In [23]:
#!pip install -q transformers datasets evaluate accelerate torch

#!pip install transformers==4.38.2 \
#    accelerate==0.27.2 \
#    tokenizers==0.15.2 \
#    datasets==2.18.0 \
#    evaluate==0.4.1 \
#    sentencepiece \
#    torch \
#    ipykernel
!pip install "accelerate==0.27.2"


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# Text Classification with IMDb
- Demonstrate basic usage of a sentiment-analysis pipeline.
- Fine-tune a pretrained text classification model using IMDb.
- Save the fine-tuned model.
- Reload the saved model with `pipeline()`.
- Classify sample texts and print results.

Let's try to use 
1. `pipeline()` for simple inference, 
2. `Trainer` for fine-tuning, 
3. `AutoTokenizer` for tokenization, and 
4. `AutoModelForSequenceClassification` classes for loading pretrained models.

We will use **DistilBERT** because it is smaller and faster than BERT, while still being a good Transformer model for teaching.

In [2]:
import numpy as np
import torch

import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, #convert text into model inputs.
    DataCollatorWithPadding, #dynamically pad inputs to the longest sequence in a batch.
    AutoModelForSequenceClassification, #load a pretrained model for text classification.
    TrainingArguments, #define training settings.
    Trainer,#train or fine-tune the model.
    pipeline,
)

checkpoint = "distilbert-base-uncased"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2026-06-11 15:18:27.424881: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-11 15:18:27.540424: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-11 15:18:27.541203: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-11 15:18:27.714214: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-11 15:18:28.831639: W tensorflow/compiler/tf

Torch: 2.12.0+cu130
CUDA available: True


## 0 Baseline: Basic sentiment pipeline before fine-tuning

This demonstrates the simplest Hugging Face workflow. 

The `pipeline()` function hides many steps: loading a model, loading a tokenizer, tokenizing the input, running inference, and converting logits into labels and scores.

In [26]:
basic_classifier = pipeline("sentiment-analysis")

test_texts = [
 "The plot is predictable and the ending is obvious, but somehow the movie is still charming, warm, and very enjoyable.",
    "The film has beautiful scenery, famous actors, and expensive special effects, but it is painfully boring from start to finish.",
    "What a masterpiece of wasted talent. Two hours of my life disappeared and I learned nothing except how bad a script can be.",
    "It is a quiet and simple film. Nothing dramatic happens, but the characters feel real and the story stays with you afterward.",
    "I can see what the director was trying to do, and I respect the ambition, but the final result just does not work.",
    "There are a few weak scenes and some awkward jokes, but overall this is a fun and surprisingly touching movie.",
    ]
    
test_results = basic_classifier(test_texts)

for text, result in zip(test_texts, test_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)



No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/home/user/ai-course/week9/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Text: The plot is predictable and the ending is obvious, but somehow the movie is still charming, warm, and very enjoyable.
Result: {'label': 'POSITIVE', 'score': 0.999855637550354}
--------------------------------------------------------------------------------
Text: The film has beautiful scenery, famous actors, and expensive special effects, but it is painfully boring from start to finish.
Result: {'label': 'NEGATIVE', 'score': 0.9990723133087158}
--------------------------------------------------------------------------------
Text: What a masterpiece of wasted talent. Two hours of my life disappeared and I learned nothing except how bad a script can be.
Result: {'label': 'NEGATIVE', 'score': 0.9998123049736023}
--------------------------------------------------------------------------------
Text: It is a quiet and simple film. Nothing dramatic happens, but the characters feel real and the story stays with you afterward.
Result: {'label': 'POSITIVE', 'score': 0.9993501305580139}
---

## 1 Load IMDb dataset

IMDb is a binary sentiment classification dataset:

- label `0` = negative,
- label `1` = positive.

To keep the notebook practical for class, we use only a small subset.

In [4]:
dataset = load_dataset("imdb")
dataset.shape

#Small subsets
#train_ds = dataset["train"].shuffle(seed=42).select(range(2000))
#test_ds = dataset["test"].shuffle(seed=42).select(range(500))

train_ds = dataset["train"] 
test_ds = dataset["test"] 


print(train_ds[0])
print(test_ds[0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## 2 Tokenization

Neural networks cannot directly process raw text. The tokenizer converts text into numerical token IDs.

1. For Transformer models, common tokenizer outputs include:
- `input_ids`: integer IDs for tokens,
- `attention_mask`: tells the model which tokens are real and which are padding.

2. We use `truncation=True` to cut very long reviews, `padding="max_length"` to make all examples the same length, and `max_length=256` to reduce memory use.

3. `batched=True` : tells Hugging Face to tokenize many examples at the same time, which is faster than inputs["texts"]
- withput `batch=True`
{   "text": "I love this movie.",
    "label": 1}
- With `batch=True`
{
    "text": [
        "I love this movie.",
        "I hate this movie.",
        "This film is okay."
    ],
    "label": [1, 0, 1]
}

In [5]:
# Load the tokenizer for the pretrained model
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [6]:
def tokenize_function(inputs):
    return tokenizer(
        inputs["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

# Remove raw text to save memory and match model input format.
# train_ds = train_ds.remove_columns(["text"])
# test_ds = test_ds.remove_columns(["text"])

# Hugging Face Trainer expects the target column to be named "labels".
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

# Not always necessary but recommand: return pytorch tensors instead of lists for model training.
#train_ds.set_format("torch")
#test_ds.set_format("torch")

print(train_ds[0])
print(train_ds[0].keys())



{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## 3 Load model and define metric
`checkpoint = "distilbert-base-uncased"`

The base DistilBERT model is used to represent and understand the sentence, but it is not directly designed for IMDb sentiment classification.

When we use `AutoModel` , as we did earlier, the model only returns hidden states — contextual vector representations of the input tokens. It does not output POSITIVE or NEGATIVE labels.

Therefore, for sentiment classification, we use `AutoModelForSequenceClassification`. This loads a Transformer model with an additional classification head. The base DistilBERT model produces contextual text representations, and the classification head maps those representations to two sentiment labels: NEGATIVE and POSITIVE.

In [7]:
# This is a 2-class classification model.
# Class 0 means NEGATIVE. Class 1 means POSITIVE.
# mapping labels to human-readable form correctly
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

#pip install scikit-learn
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Fine-tune with `Trainer`

`TrainingArguments` defines the training settings.

`Trainer` runs the training loop for us.

For classroom demo, we use small settings:
- 1 epoch
- small batch size
- evaluation at the end of each epoch
- no external logging tools



To Fine-tune the model using our specific data, we need to design some important hyperparameters:

| Hyperparameter | Meaning |
|---|---|
| `learning_rate=2e-5` | Common small learning rate for Transformer fine-tuning |
| `batch_size=8` | Small enough for a demo |
| `num_train_epochs=1` | Fast classroom demo; students may increase |
| `weight_decay=0.01` | Regularization |


In [8]:
import transformers
import accelerate

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.38.2
accelerate: 0.27.2


In [ ]:
#Tells HF how to train the model, 
training_args = TrainingArguments(
    output_dir="./imdb_training_checkpoints", #where to save checkpoints
    #eval_strategy ="epoch",
    evaluation_strategy="epoch", #evaluate the model at the end of each epoch
    save_strategy="epoch", #save the model at the end of each epoch
    learning_rate=2e-5, #the default learning rate for fine-tuning is usually between 2e-5 and 5e-5
                        #the learning rate is usually small because the model is already pretrained
    per_device_train_batch_size=8,#each device processes 8 examples at a time in training.
    per_device_eval_batch_size=8, #each device processes 8 examples at a time in testing/validation
    num_train_epochs=10,#number of epochs
    weight_decay=0.01, #Weight decay helps reduce overfitting by penalizing overly large parameter values.
    logging_steps=50,#Print or record training logs every 50 training steps.
    load_best_model_at_end=True,#After training finishes, reload the checkpoint with the best evaluation performance.
    report_to="none",#do not send training logs to any external service (like TensorBoard or Weights & Biases
)

#what model/data/metrics to use
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

### Start Training

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.288700,0.337416,0.891360
2,0.359300,0.294634,0.907760
3,0.165800,0.405484,0.905240
4,0.059600,0.557437,0.909280
5,0.088800,0.643713,0.903400
6,0.005600,0.699114,0.907080
7,0.035900,0.730378,0.906720
8,0.000100,0.807491,0.906520
9,0.014800,0.810060,0.910840
10,0.000000,0.847264,0.908840


Checkpoint destination directory ./imdb_training_checkpoints/checkpoint-3125 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_training_checkpoints/checkpoint-6250 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_training_checkpoints/checkpoint-9375 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_training_checkpoints/checkpoint-12500 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_training_checkpoints/checkpoint-15625 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_training_checkpoints/checkpoint-18750 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint de

TrainOutput(global_step=31250, training_loss=0.08698697211095877, metrics={'train_runtime': 6416.9387, 'train_samples_per_second': 38.959, 'train_steps_per_second': 4.87, 'total_flos': 1.6558424832e+16, 'train_loss': 0.08698697211095877, 'epoch': 10.0})

### Evaluate the Fine-tuned Model

In [11]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.29463401436805725, 'eval_accuracy': 0.90776, 'eval_runtime': 234.4656, 'eval_samples_per_second': 106.625, 'eval_steps_per_second': 13.328, 'epoch': 10.0}


## 5 Save and reload the fine-tuned model through `pipeline()`

This is important because the assignment asks students to use the trained model with the Hugging Face `pipeline()` function.

In [25]:
OUTPUT_DIR = "./saved_imdb_sentiment_model"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
)

print("=== Fine-tuned IMDb sentiment pipeline ===")


test_results = fine_tuned_classifier(test_texts)

for text, result in zip(test_texts, test_results):
    print("Text:", text)
    print("Result:", result)
    print("-" * 80)

=== Fine-tuned IMDb sentiment pipeline ===
Text: The plot is predictable and the ending is obvious, but somehow the movie is still charming, warm, and very enjoyable.
Result: {'label': 'POSITIVE', 'score': 0.9971229434013367}
--------------------------------------------------------------------------------
Text: The film has beautiful scenery, famous actors, and expensive special effects, but it is painfully boring from start to finish.
Result: {'label': 'NEGATIVE', 'score': 0.9952787160873413}
--------------------------------------------------------------------------------
Text: What a masterpiece of wasted talent. Two hours of my life disappeared and I learned nothing except how bad a script can be.
Result: {'label': 'NEGATIVE', 'score': 0.9922069907188416}
--------------------------------------------------------------------------------
Text: It is a quiet and simple film. Nothing dramatic happens, but the characters feel real and the story stays with you afterward.
Result: {'label': 